# test
> Run unit tests on notebooks in parallel
- order: 12

In [ ]:
#| default_exp test

In [ ]:
#| export
import time,os,sys,io,traceback,contextlib,signal,asyncio,threading
from fastcore.basics import *
from fastcore.imports import *
from fastcore.foundation import *
from fastcore.parallel import *
from fastcore.script import *
from fastcore.meta import delegates

from nbdev.config import *
from nbdev.doclinks import *
from nbdev.process import NBProcessor, nb_lang
from nbdev.frontmatter import nb_frontmatter

from fastcore.nbio import *
from execnb.shell import *

Press Ctrl-C during a hung test run to see what each notebook is executing. Each worker prints the synchronous stack and the await chains of pending asyncio tasks. An awaited coroutine can be stuck somewhere the synchronous stack doesn't show.

Workers print their diagnostics in separate blocks and exit immediately. The shell must not catch the interrupt and continue running cells. Idle workers exit silently.

In [ ]:
#| export
_cur_nb = [None]
_cur_shell = [None]

def _await_chain(t):
    "One frame per coroutine in task `t`'s await chain, deepest last: where a suspended hang actually sits"
    co = t.get_coro()
    while co is not None:
        f = getattr(co, 'cr_frame', None) or getattr(co, 'ag_frame', None) or getattr(co, 'gi_frame', None)
        if f is not None: yield f, f.f_lineno
        co = getattr(co, 'cr_await', None) or getattr(co, 'ag_await', None) or getattr(co, 'gi_yieldfrom', None)

def _int_handler(signum, frame):
    "Dump the running notebook's stack and exit; installed on SIGINT by `test_nb`"
    if _cur_nb[0] is not None:
        buf = io.StringIO()
        traceback.print_stack(frame, file=buf)
        k = _cur_shell[0]
        if k is not None:
            f = sys._current_frames().get(k._thread.ident)
            if f is not None:
                print('\n--- shell loop thread ---', file=buf)
                traceback.print_stack(f, file=buf)
            with contextlib.suppress(RuntimeError):
                for t in asyncio.all_tasks(k.loop):
                    print(f'\n{t}', file=buf)
                    buf.writelines(traceback.StackSummary.extract(_await_chain(t)).format())
        os.write(2, f'\n=== nbdev-test interrupted: {_cur_nb[0]} ===\n{buf.getvalue()}'.encode())
    os._exit(130)

In [ ]:
#| export
def _skip_cell(nb, skip_flags=None, force_flags=None):
    "Predicate for the cells of `nb` that `test_nb` leaves out"
    fm = nb_frontmatter(nb)
    if str2bool(fm.get('skip_exec', False)) or nb_lang(nb) != 'python': return lambda cell: True
    flags = set(L(skip_flags)) - set(L(force_flags))
    dflt = fm_default_eval(fm)
    def _f(cell):
        if cell.cell_type != 'code': return True
        if not does_cell_eval(cell, dflt): return True
        return flags & (getattr(cell, 'directives_', {}) or {}).keys()
    return _f

def nb_test_cells(
    fn, # Notebook path
    skip_flags=None, # Flags marking cells to skip
    force_flags=None, # Flags marking cells to always run
):
    "The cells `test_nb` would run in the notebook at `fn`"
    nb = NBProcessor(fn, rm_directives=False, process=True).nb
    skip = _skip_cell(nb, skip_flags, force_flags)
    return [c for c in nb.cells if not skip(c)]

`nb_test_cells` lists the cells selected for testing. It excludes non-Python notebooks and notebooks with `skip_exec: true` in frontmatter. Within a Python notebook, `eval` directives and skip flags determine which code cells run.

Pass `skip_flags=['notest']` to leave out cells marked `notest`:

In [ ]:
_nb = Path('../../tests/directives.ipynb')
cells = nb_test_cells(_nb, skip_flags=['notest'])
assert not any('notest' in c.source for c in cells)
len(cells), len(nb_test_cells(_nb))

In [ ]:
#| export
def test_nb(
    fn,  # file name of notebook to test
    skip_flags=None,  # list of flags marking cells to skip
    force_flags=None,  # list of flags marking cells to always run
    do_print=False,  # print completion?
    showerr=True,  # print errors to stderr?
    basepath=None,  # path to add to sys.path
    verbose=False,  # stream stdout/stderr from cells to console?
    save=False,  # write outputs back to notebook on success?
    profile:bool=None, # load the IPython profile, as `ipykernel` does? (default: `exec_profile` config key)
    cell_timeout:int=600, # seconds before each cell times out (None: no limit)
    cell_timing_min:float=None # print cells slower than this many seconds (None: no timing output)
):
    "Execute tests in notebook in `fn` except those with `skip_flags`"
    if not in_notebook() and threading.current_thread() is threading.main_thread(): signal.signal(signal.SIGINT, _int_handler)
    fn = Path(fn)
    _cur_nb[0] = fn
    if basepath: sys.path.insert(0, str(basepath))
    prev_test = os.environ.get('IN_TEST')
    if not in_notebook(): os.environ['IN_TEST'] = '1'
    try:
        nb = NBProcessor(fn, rm_directives=False, process=True).nb
        fm = nb_frontmatter(nb)
        if str2bool(fm.get('skip_exec', False)) or nb_lang(nb) != 'python': return True, 0
        _no_eval = _skip_cell(nb, skip_flags, force_flags)

        def _postproc(cell):
            elapsed = cell.metadata['execution']['total']
            if cell_timing_min is not None and elapsed > cell_timing_min: print(f'{fn.name}:{cell.id}: {elapsed:.3f}s')

        start = time.time()
        if profile is None: profile = bool(get_config(fn.parent).exec_profile)
        k = CaptureShell(fn, profile=profile)
        _cur_shell[0] = k
        if do_print: print(f'Starting {fn}')
        try:
            with working_directory(fn.parent):
                k.run_all(nb, exc_stop=True, preproc=_no_eval, postproc=_postproc, verbose=verbose, cell_timeout=cell_timeout)
                if save: write_nb(nb, fn)
                res = True
        except: 
            if showerr: sys.__stderr__.write((k.prettytb(fname=fn) or traceback.format_exc())+'\n')
            res=False
        if k.leaks and showerr: sys.__stderr__.write(f'{fn}: {len(k.leaks)} leaked task(s) survived cancellation\n')
        if do_print: print(f'- Completed {fn}')
        return res,time.time()-start
    finally:
        _cur_nb[0] = _cur_shell[0] = None
        if prev_test is None: os.environ.pop('IN_TEST', None)
        else: os.environ['IN_TEST'] = prev_test


`test_nb` runs cells on `CaptureShell`'s loop thread and waits for each result. The worker's main thread can still handle Ctrl-C while a cell is stuck.

`cell_timeout` limits execution even when synchronous code blocks the loop. A timeout fails the notebook with a `TimeoutError` identifying the cell and stuck tasks.

Failure reports use `sys.__stderr__` to bypass cell output capture. Uncancellable cell code can still be running after a timeout, with its captured `sys.stderr` still installed.


`test_nb` returns a success flag and duration. A notebook with `skip_exec: true` skips all execution and reports success. Use this for notebooks that need unavailable credentials or live services.

This test succeeds when we skip the `notest` cell:

In [ ]:
_nb = Path('../../tests/directives.ipynb')
success,duration = test_nb(_nb, skip_flags=['notest'])
assert success

The `notest` cell raises an exception. Including it makes the test fail:

In [ ]:
_nb = Path('../../tests/directives.ipynb')
success,duration = test_nb(_nb, showerr=False)
assert not success

In [ ]:
import tempfile
from fastcore.xtras import modified_env

Use `cell_timing_min` to report slow cells. Reports give the notebook name, cell ID and elapsed seconds. Try `--cell-timing-min=0.1` to start.

The `nbdev-test` command reads defaults for `cell_timeout` and `cell_timing_min` from configuration. Set them in `~/.config/nbdev/config.toml` or your project's `[tool.nbdev]` table.

`test_nb` loads the IPython profile by default, including startup files, extensions and shell configuration. This matches `ipykernel` behavior. To run without the profile, set `exec_profile = false` under `[tool.nbdev]` or pass `profile=False`:

In [ ]:
with tempfile.TemporaryDirectory() as td:
    td = Path(td)
    (td/'profile_default'/'startup').mkdir(parents=True)
    (td/'profile_default'/'startup'/'00.py').write_text('prof_x = 7\n')
    nbp = td/'prof.ipynb'
    write_nb(new_nb([mk_cell('assert prof_x==7')]), nbp)
    with modified_env(IPYTHONDIR=str(td)):
        assert     (test_nb(nbp, showerr=False))[0]
        assert not (test_nb(nbp, showerr=False, profile=False))[0]

In [ ]:
#| export
def _keep_file(
    p:Path, # filename for which to check for `indicator_fname`
    ignore_fname:str # filename that will result in siblings being ignored
) -> bool:
    "Returns False if `indicator_fname` is a sibling to `fname` else True"
    if p.exists(): return not bool(p.parent.ls().attrgot('name').filter(lambda x: x == ignore_fname))
    else: True

Set default skip flags in `tst_flags` under `[tool.nbdev]` in `pyproject.toml`. To run cells with those flags, pass them in `force_flags` to `test_nb`, or use `--flags` with `nbdev-test`.

In [ ]:
#| export
@call_parse(pos=['path'])
@delegates(nbglob_cli)
def nbdev_test(
    path:str=None,  # A notebook, directory, or full-path glob to test
    flags:str='',  # Space separated list of test flags to run that are normally ignored
    n_workers:int=None,  # Number of workers
    timing:bool=False,  # Time each notebook to see which are slow
    do_print:bool=False, # Print start and end of each notebook
    pause:float=0.01,  # Pause time (in seconds) between notebooks to avoid race conditions
    ignore_fname:str='.notest', # Filename that will result in siblings being ignored
    verbose:bool=False, # Print stdout/stderr from notebook cells?
    save:bool=False, # Write outputs back to notebooks on success?
    cell_timeout:int=None, # Seconds before each cell times out (0: no limit; default `cell_timeout` config, 600)
    cell_timing_min:float=None, # Print cells slower than this many seconds (default `cell_timing_min` config, else none)
    **kwargs
):
    "Test in parallel notebooks matching `path`, passing along `flags`"
    if path and not Path(path).exists(): kwargs['path_glob'],path = os.path.abspath(path),Path.cwd()
    cfg = get_config(Path(path).resolve() if path else None)
    cell_timeout,cell_timing_min = ifnone(cell_timeout, cfg.cell_timeout),ifnone(cell_timing_min, cfg.get('cell_timing_min'))
    skip_flags = cfg.tst_flags
    if isinstance(skip_flags, str): skip_flags = skip_flags.split()
    force_flags = flags.split()
    files = nbglob(path, as_path=True, **kwargs)
    files = [f.absolute() for f in sorted(files) if _keep_file(f, ignore_fname)]
    if len(files)==0: return print('No files were eligible for testing')

    if n_workers is None: n_workers = 0 if len(files)==1 else min(num_cpus(), 8)
    if in_notebook(): kw = {'method':'spawn'} if os.name=='nt' or sys.platform=='darwin' else {'method':'forkserver'}
    else: kw = {'method':'spawn'} if sys.platform=='darwin' else {}
    wd_pth = cfg.nbs_path
    with working_directory(wd_pth if (wd_pth and wd_pth.exists()) else os.getcwd()):
        setup = cfg.get('test_setup')
        if setup: import_obj(setup)([c for f in files for c in nb_test_cells(f, skip_flags, force_flags)])
        try:
            results = parallel(test_nb, files, skip_flags=skip_flags, force_flags=force_flags, n_workers=n_workers,
                basepath=cfg.config_path, pause=pause, do_print=do_print, verbose=verbose, save=save,
                cell_timeout=cell_timeout or None, cell_timing_min=cell_timing_min, **kw)
        except KeyboardInterrupt:
            sys.stderr.write('\nnbdev-test interrupted; in-flight notebook stacks shown above\n')
            sys.exit(130)
    passed,times = zip(*results)
    if all(passed): print("Success.")
    else: 
        _fence = '='*50
        failed = '\n\t'.join(f.name for p,f in zip(passed,files) if not p)
        sys.stderr.write(f"\nnbdev Tests Failed On The Following Notebooks:\n{_fence}\n\t{failed}\n")
        sys.exit(1)
    if timing:
        for i,t in sorted(enumerate(times), key=lambda o:o[1], reverse=True): print(f"{files[i].name}: {int(t)} secs")

Use `test_setup` for preparation shared by several notebooks, such as downloading a dataset. Running this before workers start avoids competing downloads that can exhaust cell timeouts.

Set `test_setup` under `[tool.nbdev]` to a `module:callable`. `nbdev-test` calls it once in the parent process with all cells selected by `nb_test_cells`. Its output goes directly to the console, where it can report progress.


In [ ]:
#| eval:false
nbdev_test(n_workers=0)

Success.


You can even run `nbdev-test` in non nbdev projects, for example, you can test an individual notebook like so:

```
nbdev-test ../../tests/minimal.ipynb --do-print
```

Or you can test an entire directory of notebooks filtered for only those that match a regular expression:

```
nbdev-test ../../tests --file-re '.*test.ipynb' --do-print
```

An omitted path uses the configured `nbs_path`. An existing file or directory is used directly. A path that does not exist is treated as a full-path glob filter over the working directory's tree. Quote patterns to prevent shell expansion:

```sh
nbdev-test 'nbs/api/*test*.ipynb'
```

The glob uses `fnmatch`: `*` matches directory separators and `**` has no special meaning. Use `--path-re` for regex filtering on full paths. `--file-re` matches filenames only.

## Eval

A cell's `eval` directive overrides the notebook default. Set that default in frontmatter or `metadata.nbdev`. Without either, cells run. `test_nb` uses `fastcore.nbio.does_cell_eval` for this decision.

For a slow or service-dependent notebook, set notebook-level `eval: false` and mark individual testable cells with `#| eval: true`. This differs from `skip_exec: true`, which skips the entire notebook regardless of cell directives.

Here the marked cell runs. The unmarked cell must not run:

In [ ]:
with tempfile.TemporaryDirectory() as td:
    cells = [mk_cell('---\neval: false\n---', 'raw'), mk_cell('raise Exception("unmarked: must not run")'),
        mk_cell('#| eval: true\nx = 1')]
    fn = Path(td)/'optin.ipynb'
    write_nb(new_nb(cells), fn)
    success,_ = test_nb(fn)
assert success

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()